<a href="https://colab.research.google.com/github/HudaSaffo/fashion-recommendation-system/blob/week6-candidate-outfit-construction/candidate_outfit_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6 — Candidate Retrieval and Outfit Set Construction

This notebook reuses the Week 5 rules-based RAG pipeline, retrieves real catalog candidates for each outfit slot, prunes the candidate pools, constructs combinations with bounded beam search, and returns the top-ranked outfit sets.

The active budget mode is **per product**. Each item must respect the Week 5 price constraint, while `total_outfit_price` is calculated and reported separately. Optional total-outfit budget mode is available only when explicitly enabled.

## Reused Week 5 components

The following cells reuse the Week 5 structured-intent schema, catalog normalization, rules-based query parser, knowledge retrieval, candidate ranking, and catalog-loading pipeline. They are included unchanged so this Week 6 notebook remains self-contained and reproducible.

The new Week 6 implementation begins at **Week 6 configuration**.

### Project file paths

This cell records where the refined product catalog and segmentation manifest are stored in Google Drive. Later cells use these paths to load the real project data.

In [22]:
from pathlib import Path

CATALOG_PATH = Path('/content/drive/MyDrive/catalog_refined.parquet')
SEGMENTATION_MANIFEST_PATH = Path(
    '/content/drive/MyDrive/polyvore_segmented_cache/segmentation_manifest.csv'
)

### Install dependencies

This cell installs the Python packages used by the notebook. The rules-based pipeline does not need an OpenAI API key.

In [23]:
%pip install -q pandas pyarrow faiss-cpu open_clip_torch

### Structured customer intent

This cell defines the fields used to represent a customer request: occasion, style, things to avoid, season, needed categories, color palette, and price constraints. It also checks for issues such as an unsupported category or a conflict between a desired and an avoided style. Bags are a separate category; hats and eyeglasses are accessories.

In [24]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from typing import Any

SUPPORTED_CATEGORIES = {"tops", "bottoms", "dresses", "outerwear", "shoes", "bags", "accessories"}


def _strings(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        value = [value]
    if not isinstance(value, list):
        raise ValueError("must be a string list")
    return list(dict.fromkeys(str(x).strip().lower() for x in value if str(x).strip()))


@dataclass
class PriceConstraints:
    minimum: float | None = None
    maximum: float | None = None
    currency: str | None = None

    @classmethod
    def from_dict(cls, value: Any) -> "PriceConstraints | None":
        if value in (None, {}):
            return None
        if not isinstance(value, dict):
            raise ValueError("price_constraints must be an object or null")
        minimum = float(value["minimum"]) if value.get("minimum") is not None else None
        maximum = float(value["maximum"]) if value.get("maximum") is not None else None
        if minimum is not None and minimum < 0 or maximum is not None and maximum < 0:
            raise ValueError("prices cannot be negative")
        if minimum is not None and maximum is not None and minimum > maximum:
            raise ValueError("minimum price cannot exceed maximum")
        currency = str(value["currency"]).upper() if value.get("currency") else None
        return cls(minimum, maximum, currency)


@dataclass
class ValidationIssue:
    code: str
    message: str
    severity: str = "warning"


@dataclass
class CustomerIntent:
    occasion: str | None = None
    style: list[str] = field(default_factory=list)
    avoid: list[str] = field(default_factory=list)
    season: str | None = None
    categories_needed: list[str] = field(default_factory=list)
    color_palette: list[str] = field(default_factory=list)
    price_constraints: PriceConstraints | None = None

    @classmethod
    def from_dict(cls, data: Any, strict_categories: bool = False) -> "CustomerIntent":
        if not isinstance(data, dict):
            raise ValueError("intent must be a JSON object")
        categories = _strings(data.get("categories_needed"))
        unsupported = set(categories) - SUPPORTED_CATEGORIES
        if strict_categories and unsupported:
            raise ValueError(f"unsupported categories: {', '.join(sorted(unsupported))}")
        return cls(
            occasion=str(data["occasion"]).strip().lower() if data.get("occasion") else None,
            style=_strings(data.get("style")), avoid=_strings(data.get("avoid")),
            season=str(data["season"]).strip().lower() if data.get("season") else None,
            categories_needed=categories, color_palette=_strings(data.get("color_palette")),
            price_constraints=PriceConstraints.from_dict(data.get("price_constraints")),
        )

    def validate(self) -> list[ValidationIssue]:
        issues = []
        unsupported = sorted(set(self.categories_needed) - SUPPORTED_CATEGORIES)
        if unsupported:
            issues.append(ValidationIssue("unsupported_category", f"Unsupported: {', '.join(unsupported)}"))
        conflicts = sorted(set(self.style) & set(self.avoid))
        if conflicts:
            issues.append(ValidationIssue("preference_conflict", f"Desired and avoided: {', '.join(conflicts)}"))
        if self.season and self.season not in {"spring", "summer", "autumn", "fall", "winter", "all-season"}:
            issues.append(ValidationIssue("unknown_season", f"Unknown season: {self.season}"))
        return issues

    def supported_categories(self) -> list[str]:
        return [x for x in self.categories_needed if x in SUPPORTED_CATEGORIES]

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


INTENT_JSON_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "occasion": {"type": ["string", "null"]},
        "style": {"type": "array", "items": {"type": "string"}},
        "avoid": {"type": "array", "items": {"type": "string"}},
        "season": {"type": ["string", "null"]},
        "categories_needed": {"type": "array", "items": {"type": "string"}},
        "color_palette": {"type": "array", "items": {"type": "string"}},
        "price_constraints": {"anyOf": [{"type": "null"}, {"type": "object", "additionalProperties": False,
            "properties": {"minimum": {"type": ["number", "null"]}, "maximum": {"type": ["number", "null"]}, "currency": {"type": ["string", "null"]}},
            "required": ["minimum", "maximum", "currency"]}]},
    },
    "required": ["occasion", "style", "avoid", "season", "categories_needed", "color_palette", "price_constraints"],
}

### Normalize catalog products

This cell converts a row from our known catalog into the format used by the recommendation pipeline. It uses `item_id` as the product ID, `refined_category` as the category, `refined_color` as the color, and `real_price` as the price.

In [25]:
from __future__ import annotations

import math
from typing import Any

def _clean(value: Any) -> str:
    return "" if value is None or isinstance(value, float) and math.isnan(value) else " ".join(str(value).strip().split())


def _list(value: Any) -> list[str]:
    if isinstance(value, list): return [str(x).strip().lower() for x in value]
    return [x.strip().lower() for x in _clean(value).replace("|", ",").split(",") if x.strip()]


def normalize_product(row: dict[str, Any]) -> dict[str, Any]:
    price = row.get("real_price")
    try: price = float(price) if _clean(price) else None
    except (TypeError, ValueError): price = None
    if price is not None and math.isnan(price): price = None
    return {
        "product_id": _clean(row["item_id"]),
        "title": _clean(row.get("title")).lower(),
        "description": _clean(row.get("description")).lower(),
        "category": _clean(row["refined_category"]).lower(),
        "color": _clean(row["refined_color"]).lower(),
        "material": _list(row.get("material")),
        "season": _list(row.get("season")),
        "style": _list(row.get("style")),
        "occasion": _list(row.get("occasion")),
        "price": price,
        "outfit_ids": [],
    }

### Retrieve fashion knowledge

This cell builds searchable knowledge from style rules, category and color rules, products, and existing outfits. For each customer query, it retrieves relevant context. The current notebook uses lexical (word-based) retrieval unless an embedding backend is configured.

In [26]:
from __future__ import annotations

import math, re
from collections import Counter
from dataclasses import dataclass
from typing import Any, Protocol

TOKEN = re.compile(r"[a-z0-9]+")

STYLE_GUIDES = [
    ("style", "Smart casual balances polished pieces with relaxed materials; it suits elegant events that are not too formal."),
    ("season", "Summer outfits favor breathable cotton or linen, light colors, and lighter layers."),
    ("occasion", "A dinner outfit can combine a refined top, bottom, shoes, and an optional accessory."),
    ("style", "Formal styling favors tailoring and dress shoes; casual styling favors denim, knits, and sneakers."),
]
CATEGORY_RULES = [
    ("compatibility", "A separates outfit uses tops, bottoms, shoes, and optionally bags, accessories, or outerwear."),
    ("compatibility", "A dress from dresses replaces tops and bottoms; combine it with shoes and optionally bags, accessories, or outerwear."),
]
COLOR_RULES = [
    ("color", "Neutral palettes include black, white, cream, beige, grey, navy, and brown."),
    ("color", "Light warm palettes include cream, beige, ivory, tan, camel, peach, coral, and gold."),
    ("color", "Analogous or neutral colors are easy to combine; use an accessory for a controlled accent."),
]


@dataclass
class KnowledgeChunk:
    chunk_id: str
    source: str
    text: str
    metadata: dict[str, Any]
    score: float = 0.0

    def to_dict(self): return {"chunk_id": self.chunk_id, "source": self.source, "text": self.text, "score": round(self.score, 4)}


class EmbeddingBackend(Protocol):
    def rank(self, query: str, chunks: list[KnowledgeChunk], top_k: int) -> list[KnowledgeChunk]: ...


class KnowledgeRetriever:
    def __init__(self, products, outfits=None, embedding_backend: EmbeddingBackend | None = None):
        self.embedding_backend = embedding_backend
        self.last_diagnostics = {"retrieval_mode": "lexical", "embedding_fallback_reason": None}
        self.chunks = []
        for i, (source, text) in enumerate(STYLE_GUIDES + CATEGORY_RULES + COLOR_RULES):
            self.chunks.append(KnowledgeChunk(f"rule:{i}", source, text, {}))
        for p in products:
            text = " ".join([p["title"], p["description"], p["category"], p["color"], *p["material"], *p["season"], *p["style"], *p["occasion"]])
            self.chunks.append(KnowledgeChunk(f"product:{p['product_id']}", "product", text, {"product_id": p["product_id"]}))
        for outfit in outfits or []:
            text = " ".join(str(outfit.get(k, "")) for k in ("name", "description", "style", "occasion", "season", "colors"))
            self.chunks.append(KnowledgeChunk(f"outfit:{outfit.get('outfit_id')}", "outfit", text, dict(outfit)))

    def retrieve(self, query: str, top_k: int = 8) -> list[KnowledgeChunk]:
        if self.embedding_backend:
            try:
                ranked = self.embedding_backend.rank(query, self.chunks, top_k)
                self.last_diagnostics = {"retrieval_mode": "week4_semantic", "embedding_fallback_reason": None}
                return ranked
            except Exception as exc:
                self.last_diagnostics = {
                    "retrieval_mode": "lexical",
                    "embedding_fallback_reason": type(exc).__name__,
                }
        else:
            self.last_diagnostics = {"retrieval_mode": "lexical", "embedding_fallback_reason": "not_configured"}
        q = Counter(TOKEN.findall(query.lower()))
        tokenized = [set(TOKEN.findall(c.text.lower())) for c in self.chunks]
        scored = []
        for chunk in self.chunks:
            words = Counter(TOKEN.findall(chunk.text.lower()))
            overlap = 0.0
            for token, count in q.items():
                document_frequency = sum(token in tokens for tokens in tokenized)
                if document_frequency and words[token]:
                    overlap += min(count, words[token]) * (1.0 + math.log(1 + len(self.chunks) / document_frequency))
            phrase = sum(1.5 for token in q if token in words)
            scored.append(KnowledgeChunk(chunk.chunk_id, chunk.source, chunk.text, chunk.metadata, overlap + phrase))
        positive = sorted((x for x in scored if x.score > 0), key=lambda x: (-x.score, x.chunk_id))
        return positive[:top_k]


class Week4EmbeddingBackend:

    def __init__(self, artifact_dir, index_type="text", model_name="ViT-B-32", pretrained="laion2b_s34b_b79k"):
        from pathlib import Path
        import faiss
        import pandas as pd

        artifact_dir = Path(artifact_dir)
        index_path = artifact_dir / f"{index_type}.index"
        catalog_path = artifact_dir / "retrieval_catalog.parquet"
        if not index_path.is_file() or not catalog_path.is_file():
            raise FileNotFoundError("Week 4 index or retrieval catalog is missing")
        self.index = faiss.read_index(str(index_path))
        catalog = pd.read_parquet(catalog_path)
        id_column = next((x for x in ("item_id", "product_id", "id") if x in catalog.columns), None)
        if id_column is None or len(catalog) != self.index.ntotal:
            raise ValueError("Week 4 catalog/index alignment is invalid")
        self.product_ids = catalog[id_column].astype(str).tolist()
        self.model_name, self.pretrained = model_name, pretrained
        self._model = self._tokenizer = None

    def rank(self, query, chunks, top_k):
        import numpy as np
        import open_clip, torch

        if self._model is None:
            self._model, _, _ = open_clip.create_model_and_transforms(self.model_name, pretrained=self.pretrained)
            self._model.eval()
            self._tokenizer = open_clip.get_tokenizer(self.model_name)
        with torch.inference_mode():
            vector = self._model.encode_text(self._tokenizer([query])).float().cpu().numpy()
        vector = vector / np.clip(np.linalg.norm(vector, axis=1, keepdims=True), 1e-12, None)
        product_chunks = {c.metadata.get("product_id"): c for c in chunks if c.source == "product"}
        k = min(max(top_k * 2, top_k), self.index.ntotal)
        scores, positions = self.index.search(np.ascontiguousarray(vector, dtype=np.float32), k)
        ranked = []
        for score, position in zip(scores[0], positions[0]):
            if position < 0: continue
            chunk = product_chunks.get(self.product_ids[int(position)])
            if chunk:
                ranked.append(KnowledgeChunk(chunk.chunk_id, chunk.source, chunk.text, chunk.metadata, float(score)))
            if len(ranked) >= top_k: break
        return ranked

### Understand the customer query

This cell turns customer language into structured intent. Our active pipeline uses the rules parser. The parser recognizes requests such as “summer dinner,” “not too formal,” a specific category, a color, or a price limit.

In [27]:
from __future__ import annotations

import json, re
from typing import Protocol


SYSTEM_PROMPT = """You extract fashion customer intent. Use the retrieved context as knowledge, not as customer preferences.
Return only JSON matching the supplied schema. Preserve explicit requirements. Put desired attributes in style/colors and negative requirements in avoid. Use null or [] when unknown. Never invent a preference."""


class LLMBackend(Protocol):
    def extract(self, query: str, context: str, schema: dict) -> dict: ...


class OpenAIBackend:
    def __init__(self, api_key: str, model: str): self.api_key, self.model = api_key, model
    def extract(self, query, context, schema):
        from openai import OpenAI
        response = OpenAI(api_key=self.api_key).responses.create(
            model=self.model, input=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Context:\n{context}\n\nCustomer query:\n{query}"}],
            text={"format": {"type": "json_schema", "name": "fashion_intent", "strict": True, "schema": schema}},
        )
        return json.loads(response.output_text)


class QueryParser:
    def __init__(self, llm: LLMBackend | None = None):
        self.llm = llm
        self.last_diagnostics = {"parser_mode": "rules_fallback", "llm_fallback_reason": "not_configured"}

    def parse(self, query: str, chunks=()) -> tuple[CustomerIntent, str]:
        context = "\n".join(f"[{x.source}] {x.text}" for x in chunks)
        if self.llm:
            try:
                intent = CustomerIntent.from_dict(self.llm.extract(query, context, INTENT_JSON_SCHEMA))
                self.last_diagnostics = {"parser_mode": "llm", "llm_fallback_reason": None}
                return intent, "llm"
            except Exception as exc:
                self.last_diagnostics = {"parser_mode": "rules_fallback", "llm_fallback_reason": type(exc).__name__}
        else:
            self.last_diagnostics = {"parser_mode": "rules_fallback", "llm_fallback_reason": "not_configured"}
        return self._rules(query), "rules_fallback"

    def _rules(self, query: str) -> CustomerIntent:
        text = " ".join(query.lower().split())
        avoid, spans = [], []
        pattern = re.compile(r"\b(?:not|avoid|no|without)\s+((?:too\s+)?[a-z]+(?:\s+[a-z]+){0,2}?)(?=\s+(?:and\s+(?:no|avoid|without|not)|but|for|with|under|below)\b|[,.;!?]|$)")
        for match in pattern.finditer(text):
            avoid.append(match.group(1).strip())
            spans.append(match.span())
        positive_text = text
        for start, end in reversed(spans): positive_text = positive_text[:start] + " " + positive_text[end:]

        styles = []
        if re.search(r"\bsmart[ -]casual\b", positive_text): styles.append("smart casual")
        for value in ("elegant", "casual", "formal", "minimal", "sporty", "bohemian", "classic"):
            if re.search(rf"\b{re.escape(value)}\b", positive_text):
                if value != "casual" or not re.search(r"\bsmart[ -]casual\b", positive_text): styles.append(value)
        if "elegant" in styles and any("formal" in x for x in avoid) and "smart casual" not in styles: styles.append("smart casual")
        if re.search(r"\bpolished\b", positive_text) and "classic" not in styles: styles.append("classic")
        season = next((x for x in ("spring", "summer", "autumn", "fall", "winter") if re.search(rf"\b{x}\b", positive_text)), None)
        categories = []
        mapping = {"tops": ("top", "shirt", "blouse"), "bottoms": ("bottom", "trouser", "pants", "skirt", "jeans"), "dresses": ("dress", "dresses"), "outerwear": ("jacket", "coat", "outerwear"), "shoes": ("shoe", "sneaker", "heel", "loafer", "boot"), "bags": ("bag", "handbag", "purse", "tote", "clutch"), "accessories": ("accessory", "accessories", "hat", "cap", "eyeglasses", "glasses", "sunglasses", "belt", "jewelry", "jewellery", "necklace", "earring")}
        for category, terms in mapping.items():
            if any(re.search(rf"\b{re.escape(term)}s?\b", positive_text) for term in terms): categories.append(category)
        outfit_words = ("outfit", "something", "look", "wear")
        if not categories and any(re.search(rf"\b{x}\b", positive_text) for x in outfit_words): categories = ["tops", "bottoms", "shoes", "bags", "accessories"]
        colors = [x for x in ("neutral", "light", "warm", "bright", "black", "white", "cream", "beige", "navy", "blue", "red", "green", "pink") if re.search(rf"\b{x}\b", positive_text)]
        if season == "summer" and "elegant" in styles and not colors: colors = ["neutral", "light", "warm"]
        maximum = None
        price = re.search(r"(?:under|below|up to|max(?:imum)?(?: of)?)\s*([$€£])?\s*(\d+(?:\.\d+)?)", text)
        currency = {"$": "USD", "€": "EUR", "£": "GBP"}.get(price.group(1)) if price else None
        if price: maximum = float(price.group(2))
        occasion = next((x for x in ("summer dinner", "dinner", "wedding", "work", "office", "party", "date", "travel", "everyday") if x in text), None)
        data = {"occasion": occasion, "style": styles, "avoid": avoid, "season": season, "categories_needed": categories, "color_palette": colors,
                "price_constraints": {"minimum": None, "maximum": maximum, "currency": currency} if maximum is not None else None}
        return CustomerIntent.from_dict(data)

### Filter and rank product candidates

This cell selects products from the requested categories, applies restrictions such as season, price, avoided terms, sneakers, and cotton, then ranks the remaining products by style, occasion, color, and related outfits. It also filters obvious category mistakes and clearly unsuitable office or dinner items. A price limit currently applies to each candidate product, not to the total price of a complete outfit.

In [28]:
from __future__ import annotations

import re
from collections import defaultdict

NEUTRAL = {"black", "white", "cream", "beige", "grey", "gray", "navy", "brown", "ivory", "tan"}
LIGHT = {"white", "cream", "beige", "ivory", "pastel", "light blue", "peach"}
WARM = {"cream", "beige", "ivory", "tan", "camel", "peach", "coral", "gold", "brown"}
BRIGHT = {"bright", "neon", "vivid", "red", "orange", "yellow", "lime", "hot pink", "fuchsia"}

STYLE_ALIASES = {"smart-casual": "smart casual"}


def _matches_color(color: str, requested: list[str]) -> bool:
    for item in requested:
        if item in color: return True
        if item == "neutral" and color in NEUTRAL: return True
        if item == "light" and color in LIGHT: return True
        if item == "warm" and color in WARM: return True
        if item == "bright" and color in BRIGHT: return True
    return False


def _norm(value):
    value = " ".join(str(value).lower().replace("-", " ").split())
    return STYLE_ALIASES.get(value, value)


def _taxonomy_hits(requested, actual):
    actual_set = {_norm(x) for x in actual}
    return [x for x in requested if _norm(x) in actual_set]


def _occasion_hits(requested, actual):
    aliases = {
        "office": {"office", "workwear", "work", "business"},
        "work": {"office", "workwear", "work", "business"},
        "summer dinner": {"summer dinner", "dinner", "evening out"},
        "dinner": {"dinner", "evening out"},
    }
    actual_set = {_norm(x) for x in actual}
    return [
        wanted for wanted in requested
        if actual_set & aliases.get(_norm(wanted), {_norm(wanted)})
    ]


def _excluded(product, avoided):
    haystack = " ".join([product["title"], product["description"], product["color"], *product["style"], *product["material"], *product["occasion"]])
    for raw in avoided:
        term = re.sub(r"^too\s+", "", _norm(raw))
        if term in {"bright", "bright color", "bright colors"} and product["color"] in BRIGHT: return True
        if term in {"high heel", "high heels"} and re.search(r"\b(?:high\s+)?heels?\b", haystack): return True
        if term and re.search(rf"\b{re.escape(term)}\b", haystack): return True
    return False

def _type_conflict(product):
    title = product["title"]
    category = product["category"]
    if category == "shoes":
        if title.startswith("a stylish pair of accessories:"):
            return True
        if re.search(
            r"\b(?:belts?|sunglasses?|eyeglasses?|glasses|eyewear|frames)\b",
            title,
        ):
            return True
    if category == "dresses":
        looks_like_separate = re.search(
            r"\b(?:shirt|blouse|top|skirt|shorts|pants|trousers)\b",
            title,
        )
        mentions_dress = re.search(r"\b(?:dress|gown)\b", title)
        if looks_like_separate and not mentions_dress:
            return True
    return False

def _query_requirements(query):
    text = _norm(query)
    return {
        "sneakers": bool(re.search(r"\b(?:sneakers?|trainers?)\b", text)),
        "cotton": bool(re.search(r"\bcotton\b", text)),
        "denim": bool(re.search(r"\bdenim\b", text)),
    }

def _occasion_type_conflict(product, intent):
    title = product["title"]
    obviously_sporty = re.search(
        r"\b(?:bras?|bralettes?|swimwear|bikini|workout|gym)\b", title
    )
    if intent.occasion in {"office", "work", "dinner", "summer dinner"}:
        if obviously_sporty:
            return True
    if intent.occasion in {"office", "work"}:
        if re.search(r"\b(?:sweatshirt|hoodie|plimsolls)\b", title):
            return True
    if intent.occasion in {"dinner", "summer dinner"}:
        if re.search(
            r"\b(?:sweatshirt|hoodie|ripped|distressed|hair elastics?)\b",
            title,
        ):
            return True
    return False

class CatalogCandidateRetriever:
    def __init__(self, products, outfits=None, min_relevance_score=2.0):
        self.products = products
        self.min_relevance_score = float(min_relevance_score)
        self.outfits = {str(x.get("outfit_id")): x for x in (outfits or []) if x.get("outfit_id")}
        self.product_outfits = defaultdict(set)
        for outfit_id, outfit in self.outfits.items():
            for product_id in outfit.get("product_ids", []): self.product_outfits[str(product_id)].add(outfit_id)

    def relevant_outfits(self, intent):
        relevant = set()
        for outfit_id, outfit in self.outfits.items():
            score = 0
            style = _style_values(outfit.get("style"))
            occasion = _listish(outfit.get("occasion"))
            colors = _listish(outfit.get("colors"))
            season = _listish(outfit.get("season"))
            if _taxonomy_hits(intent.style, style): score += 2
            if _occasion_hits([intent.occasion] if intent.occasion else [], occasion): score += 2
            if intent.season and _norm(intent.season) in {_norm(x) for x in season}: score += 1
            if intent.color_palette and any(_matches_color(color, intent.color_palette) for color in colors): score += 1
            if score >= 2: relevant.add(outfit_id)
        return relevant

    def retrieve(self, intent, per_category=5, min_relevance_score=None, relevant_outfit_ids=None, query=""):
        categories = intent.supported_categories()
        if not categories: return {}
        threshold = self.min_relevance_score if min_relevance_score is None else float(min_relevance_score)
        relevant_outfit_ids = set(relevant_outfit_ids or []) | self.relevant_outfits(intent)
        self.last_relevant_outfit_ids = sorted(relevant_outfit_ids)
        needs = _query_requirements(query)
        grouped = defaultdict(list)
        for p in self.products:
            if (p["category"] not in categories or _type_conflict(p) or _occasion_type_conflict(p, intent)): continue
            if intent.season and p["season"] and intent.season not in p["season"] and "all-season" not in p["season"]: continue
            pc = intent.price_constraints
            if pc and pc.minimum is not None and (p["price"] is None or p["price"] < pc.minimum): continue
            if pc and pc.maximum is not None and (p["price"] is None or p["price"] > pc.maximum): continue
            if _excluded(p, intent.avoid): continue
            if needs["sneakers"] and p["category"] == "shoes":
                if not re.search(r"\b(?:sneakers?|trainers?)\b", p["title"]): continue
            if needs["cotton"] and p["category"] in {"tops", "bottoms", "dresses", "outerwear"}:
                if "cotton" not in p["material"]: continue
            reasons, score = [], 0.0
            for label, requested, actual, weight in (("style", intent.style, p["style"], 3),):
                hits = _taxonomy_hits(requested, actual)
                if hits: score += weight * len(hits); reasons.append(f"{label}: {', '.join(hits)}")
            occasion_hits = _occasion_hits([intent.occasion] if intent.occasion else [], p["occasion"])
            if occasion_hits: score += 3 * len(occasion_hits); reasons.append(f"occasion: {', '.join(occasion_hits)}")
            if intent.color_palette and _matches_color(p["color"], intent.color_palette): score += 1; reasons.append(f"color: {p['color']}")
            if needs["sneakers"] and p["category"] == "shoes":
              score += 3
              reasons.append("type: sneakers")
            if needs["cotton"] and "cotton" in p["material"]:
              score += 3
              reasons.append("material: cotton")
            if needs["denim"] and (
              "denim" in p["material"] or re.search(r"\bdenim\b", p["title"])):
              score += 2
              reasons.append("denim")
            memberships = set(p["outfit_ids"]) | self.product_outfits.get(p["product_id"], set())
            relevant_memberships = memberships & relevant_outfit_ids
            if relevant_memberships: score += 0.5; reasons.append(f"relevant outfit: {sorted(relevant_memberships)[0]}")
            if score >= threshold:
                grouped[p["category"]].append({"product_id": p["product_id"], "title": p["title"], "score": round(score, 3), "price": p["price"], "reasons": reasons})
        return {cat: sorted(grouped.get(cat, []), key=lambda x: (-x["score"], x["product_id"]))[:per_category] for cat in categories}


def _listish(value):
    if value is None: return []
    if isinstance(value, list): return [_norm(x) for x in value]
    return [_norm(x) for x in re.split(r"[,|]", str(value)) if _norm(x)]


def _style_values(value):
    text = _norm(" ".join(value) if isinstance(value, list) else value or "")
    found = []
    if re.search(r"\bsmart casual\b", text):
        found.append("smart casual")
        text = re.sub(r"\bsmart casual\b", " ", text)
    for style in ("elegant", "casual", "formal", "minimal", "sporty", "bohemian", "classic"):
        if re.search(rf"\b{style}\b", text): found.append(style)
    return found

### Connect the pipeline

This cell joins the three main stages: retrieve relevant knowledge, parse the customer’s intent, and retrieve catalog products. `pipeline.run(query)` returns the structured intent, retrieved knowledge, validation issues, diagnostics, and ranked candidates.

In [29]:
from __future__ import annotations

class FashionRAGPipeline:
    def __init__(self, catalog, outfits=None, llm=None, embedding_backend=None, min_relevance_score=2.0):
        self.catalog = catalog
        self.outfits = outfits or []
        self.knowledge = KnowledgeRetriever(catalog, self.outfits, embedding_backend)
        self.parser = QueryParser(llm)
        self.ranker = CatalogCandidateRetriever(catalog, self.outfits, min_relevance_score)

    def run(self, query: str, knowledge_top_k=8, candidates_per_category=5, min_relevance_score=None):
        if not isinstance(query, str) or not query.strip(): raise ValueError("query must be non-empty")
        knowledge = self.knowledge.retrieve(query, knowledge_top_k)
        intent, parser_used = self.parser.parse(query, knowledge)
        issues = intent.validate()
        retrieved_outfit_ids = {str(x.metadata.get("outfit_id")) for x in knowledge if x.source == "outfit" and x.metadata.get("outfit_id")}
        relevant_outfit_ids = retrieved_outfit_ids & self.ranker.relevant_outfits(intent)
        candidates = self.ranker.retrieve(intent, candidates_per_category, min_relevance_score,relevant_outfit_ids, query=query)
        diagnostics = {**self.knowledge.last_diagnostics, **self.parser.last_diagnostics,
            "embedding_setup_fallback_reason": getattr(self, "embedding_setup_fallback_reason", None),
            "catalog_product_count": len(self.catalog),
            "relevant_outfit_ids": getattr(self.ranker, "last_relevant_outfit_ids", sorted(relevant_outfit_ids))}
        return {"query": query, "retrieved_knowledge": [x.to_dict() for x in knowledge], "intent": intent.to_dict(),
                "validation_issues": [x.__dict__ for x in issues], "parser": parser_used, "diagnostics": diagnostics, "candidates": candidates}

### Load the real catalog and existing outfits

This cell reads the refined catalog from Google Drive. Rows with the same `set_id` are grouped into existing outfits; products are identified by `item_id`. It then creates one normalized record per product and builds the rules-based pipeline. The segmentation manifest is loaded for project context but is not used to rank products in this cell.

In [30]:
import json
import re
import pandas as pd

MIN_RELEVANCE_SCORE = 2.0

raw_catalog = pd.read_parquet(CATALOG_PATH)
segmentation_manifest = pd.read_csv(SEGMENTATION_MANIFEST_PATH)

print(f'Raw catalog rows: {len(raw_catalog):,}')
print(f'Segmentation-manifest rows: {len(segmentation_manifest):,}')
print('Catalog columns:', raw_catalog.columns.tolist())

item_id_column = 'item_id'
set_id_column = 'set_id'

raw_catalog[item_id_column] = raw_catalog[item_id_column].astype(str).str.strip()
raw_catalog[set_id_column] = raw_catalog[set_id_column].astype(str).str.strip()
raw_catalog = raw_catalog[
    raw_catalog[item_id_column].ne('') & raw_catalog[set_id_column].ne('')
].copy()

raw_catalog = raw_catalog.drop_duplicates(
    subset=[set_id_column, item_id_column], keep='first'
).copy()

print('Refined categories:')
print(raw_catalog['refined_category'].value_counts(dropna=False))
print('Sample refined colors:')
print(raw_catalog['refined_color'].value_counts(dropna=False).head(20))


def unique_values(series):
    """Collect normalized comma/pipe-separated metadata without inventing values."""
    values = []
    for raw in series.dropna():
        for value in re.split(r'[,|]', str(raw)):
            value = ' '.join(value.strip().lower().split())
            if value and value not in values:
                values.append(value)
    return values


outfits = []
for set_id, group in raw_catalog.groupby(set_id_column, sort=False):
    product_ids = list(dict.fromkeys(group[item_id_column].astype(str)))
    if len(product_ids) < 2:
        continue

    outfit = {
        'outfit_id': str(set_id),
        'product_ids': product_ids,
    }
    outfit['style'] = unique_values(group['style'])
    outfit['occasion'] = unique_values(group['occasion'])
    outfit['season'] = unique_values(group['season'])
    outfit['colors'] = unique_values(group['refined_color'])
    categories = unique_values(group['refined_category'])
    outfit['name'] = f'Catalog outfit {set_id}'
    outfit['description'] = 'Existing outfit containing: ' + ', '.join(categories)
    outfits.append(outfit)

print(f'Created {len(outfits):,} existing outfits from {set_id_column}.')
if outfits:
    print('Sample outfit:')
    print(json.dumps(outfits[0], indent=2))

product_rows = raw_catalog.drop_duplicates(item_id_column, keep='first').copy()
catalog = [normalize_product(row) for row in product_rows.to_dict('records')]

pipeline = FashionRAGPipeline(
    catalog,
    outfits=outfits,
    min_relevance_score=MIN_RELEVANCE_SCORE,
)
CATALOG_USED = str(CATALOG_PATH)

print(f'Loaded {len(catalog):,} unique products from {CATALOG_USED}')
print('Sample normalized product:')
print(json.dumps(catalog[0], indent=2))


Raw catalog rows: 3,673
Segmentation-manifest rows: 3,673
Catalog columns: ['set_id', 'item_id', 'item_index_in_set', 'url_name', 'title', 'description', 'category', 'color', 'material', 'season', 'style', 'occasion', 'fit', 'formality', 'real_price', 'price_bucket', 'refined_category', 'refined_color']
Refined categories:
refined_category
accessories    867
shoes          789
bags           668
bottoms        454
tops           413
outerwear      256
dresses        226
Name: count, dtype: int64
Sample refined colors:
refined_color
black            617
blue             409
white            360
gold             329
beige            318
pink             313
red              237
brown            218
grey             191
green            186
silver           143
multi/neutral    114
yellow            90
purple            81
orange            67
Name: count, dtype: int64
Created 1,109 existing outfits from set_id.
Sample outfit:
{
  "outfit_id": "210750761",
  "product_ids": [
    "15424972

## Week 6 configuration

These settings bound the search so the notebook does not create the full Cartesian product. Clothing slots keep up to 10 candidates; bags and accessories keep up to 5.

In [31]:
from dataclasses import dataclass, field
from itertools import combinations
from typing import Any


@dataclass
class OutfitConstructionConfig:
    retrieval_pool_size: int = 20
    clothing_limit: int = 10
    bag_limit: int = 5
    accessory_limit: int = 5
    beam_width: int = 60
    max_combinations_evaluated: int = 2000
    top_n: int = 5
    total_budget_mode: bool = False
    total_budget: float | None = None
    weights: dict[str, float] = field(default_factory=lambda: {
        "item_relevance": 0.22,
        "category_completeness": 0.15,
        "color_compatibility": 0.18,
        "style_compatibility": 0.17,
        "occasion_compatibility": 0.13,
        "season_compatibility": 0.08,
        "existing_outfit_evidence": 0.05,
        "price_efficiency": 0.02,
        "duplicate_penalty": 0.10,
})


WEEK6_CONFIG = OutfitConstructionConfig()

## Outfit slot planning

A dress replaces tops and bottoms. A separates outfit uses tops, bottoms and shoes. Bags, accessories and outerwear are optional unless explicitly requested. A single-product request such as “white sneakers” remains a shoes-only retrieval task.

In [32]:
class OutfitSlotPlanner:
    BAG_TERMS = re.compile(r"\b(?:bags?|handbags?|purses?|totes?|clutches?)\b")
    ACCESSORY_TERMS = re.compile(
        r"\b(?:accessory|accessories|hats?|caps?|eyeglasses|glasses|sunglasses|"
        r"belts?|jewelry|jewellery|necklaces?|earrings?)\b"
    )
    OUTERWEAR_TERMS = re.compile(r"\b(?:outerwear|jackets?|coats?)\b")

    def plan(self, query: str, intent: CustomerIntent) -> dict[str, Any]:
        text = _norm(query)
        requested = list(dict.fromkeys(intent.supported_categories()))
        explicit_outfit = bool(re.search(r"\b(?:outfit|look|something|wear)\b", text))
        explicit_bag = bool(self.BAG_TERMS.search(text))
        explicit_accessory = bool(self.ACCESSORY_TERMS.search(text))
        explicit_outerwear = bool(self.OUTERWEAR_TERMS.search(text))

        if "dresses" in requested:
            required = ["dresses", "shoes"]
            optional = []
        elif explicit_outfit or any(x in requested for x in ("tops", "bottoms")):
            required = ["tops", "bottoms", "shoes"]
            optional = []
        else:
            required = requested or ["tops", "bottoms", "shoes"]
            optional = []

        if explicit_bag:
            required.append("bags")
        elif "bags" in requested or explicit_outfit:
            optional.append("bags")

        if explicit_accessory:
            required.append("accessories")
        elif "accessories" in requested or explicit_outfit:
            optional.append("accessories")

        if explicit_outerwear:
            required.append("outerwear")
        elif "outerwear" in requested or (intent.season == "winter" and explicit_outfit):
            optional.append("outerwear")

        required = list(dict.fromkeys(required))
        optional = [x for x in dict.fromkeys(optional) if x not in required]
        if "dresses" in required:
            required = [x for x in required if x not in {"tops", "bottoms"}]
            optional = [x for x in optional if x not in {"tops", "bottoms"}]
        return {"required": required, "optional": optional, "all": required + optional}

## Candidate pruning

The Week 5 candidate retriever supplies category-aware products. This stage keeps only the configured number per slot and removes near-duplicates using normalized title, refined category and refined color. Availability, gender and size filters are skipped because those reliable fields are not present in the known catalog.

In [33]:
GENERATED_TITLE_PREFIX = re.compile(
    r"^a stylish pair of (?:tops|bottoms|shoes|bags|accessories):\s*"
)


def _singular_token(token: str) -> str:
    special = {
        "accessories": "accessory", "dresses": "dress", "shoes": "shoe",
        "glasses": "glass", "sunglasses": "sunglass", "jeans": "jean",
    }
    if token in special:
        return special[token]
    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"
    if token.endswith("s") and not token.endswith("ss") and len(token) > 3:
        return token[:-1]
    return token


def _normalized_product_title(value: str) -> str:
    title = GENERATED_TITLE_PREFIX.sub("", _norm(value))
    tokens = re.findall(r"[a-z0-9]+", title)
    return " ".join(_singular_token(token) for token in tokens)


def _duplicate_key(product: dict[str, Any]) -> tuple[str, str, str]:
    return (
        _normalized_product_title(product.get("title", "")),
        _norm(product.get("category", "")),
        _norm(product.get("color", "")),
    )


class CandidatePruner:
    def __init__(self, config: OutfitConstructionConfig):
        self.config = config

    def limit_for(self, slot: str) -> int:
        if slot == "bags": return self.config.bag_limit
        if slot == "accessories": return self.config.accessory_limit
        return self.config.clothing_limit

    def prune(self, slot_candidates, slots, product_by_id):
        before, after, pruned = {}, {}, {}
        for slot in slots["all"]:
            candidates = list(slot_candidates.get(slot, []))
            before[slot] = len(candidates)
            kept, seen = [], set()
            for candidate in candidates:
                product = product_by_id[str(candidate["product_id"])]
                key = _duplicate_key(product)
                if key in seen:
                    continue
                seen.add(key)
                merged = {**product, "week5_score": float(candidate.get("score", 0.0)),
                          "week5_reasons": list(candidate.get("reasons", []))}
                kept.append(merged)
                if len(kept) >= self.limit_for(slot):
                    break
            after[slot] = len(kept)
            pruned[slot] = kept
        return pruned, before, after

## Outfit-level scoring

Each combination is scored for item relevance, completeness, color, style, occasion, season, shared existing-outfit evidence, price efficiency and duplicate risk. The weights are defined in the configuration cell.

In [34]:
def _pairwise_ratio(items, predicate):
    pairs = list(combinations(items, 2))
    return 0.5 if not pairs else sum(predicate(a, b) for a, b in pairs) / len(pairs)


def _metadata_compatibility(a, b, key):
    left, right = set(a.get(key, [])), set(b.get(key, []))
    if not left or not right:
        return 0.5
    return float(bool(left & right))


class OutfitScorer:
    def __init__(self, config, product_outfits):
        self.config = config
        self.product_outfits = product_outfits

    def score(self, items, required_slots, intent):
        products = list(items.values())
        relevance = sum(min(p.get("week5_score", 0.0) / 8.0, 1.0) for p in products) / max(len(products), 1)
        completeness = sum(slot in items for slot in required_slots) / max(len(required_slots), 1)

        def color_pair(a, b):
            ca, cb = _norm(a.get("color", "")), _norm(b.get("color", ""))
            if not ca or not cb:
                return 0.5
            return float(ca == cb or ca in NEUTRAL or cb in NEUTRAL or "multi/neutral" in {ca, cb})

        color = _pairwise_ratio(products, color_pair)
        style = _pairwise_ratio(products, lambda a, b: _metadata_compatibility(a, b, "style"))

        occasion_values = []
        for product in products:
            metadata = product.get("occasion", [])
            if not intent.occasion:
                occasion_values.append(0.5)
            elif not metadata:
                occasion_values.append(0.5)
            else:
                occasion_values.append(float(bool(_occasion_hits([intent.occasion], metadata))))
        occasion = sum(occasion_values) / max(len(occasion_values), 1)

        season_values = []
        for product in products:
            metadata = product.get("season", [])
            if not intent.season:
                season_values.append(0.5)
            elif not metadata:
                season_values.append(0.5)
            else:
                season_values.append(float(intent.season in metadata or "all-season" in metadata))
        season = sum(season_values) / max(len(season_values), 1)

        shared_outfit = 0.0
        if len(products) > 1:
            shared_outfit = _pairwise_ratio(products, lambda a, b: float(bool(
                self.product_outfits.get(str(a["product_id"]), set()) &
                self.product_outfits.get(str(b["product_id"]), set())
            )))

        keys = [_duplicate_key(p) for p in products]
        duplicate_penalty = 1.0 - len(set(keys)) / max(len(keys), 1)
        prices = [p["price"] for p in products if p.get("price") is not None]
        total_price = round(sum(prices), 2)
        price_efficiency = 1.0 / (1.0 + total_price / 200.0)

        breakdown = {
            "item_relevance": relevance,
            "category_completeness": completeness,
            "color_compatibility": color,
            "style_compatibility": style,
            "occasion_compatibility": occasion,
            "season_compatibility": season,
            "existing_outfit_evidence": shared_outfit,
            "price_efficiency": price_efficiency,
            "duplicate_penalty": duplicate_penalty,
        }
        positive = sum(self.config.weights[k] * breakdown[k] for k in breakdown if k != "duplicate_penalty")
        final = max(0.0, 10.0 * (positive - self.config.weights["duplicate_penalty"] * duplicate_penalty))
        reasons = ["complete required slots"] if completeness == 1 else []
        if color >= 0.7: reasons.append("compatible color palette")
        if style >= 0.7: reasons.append("compatible product styles")
        if occasion >= 0.7 and intent.occasion: reasons.append(f"fits {intent.occasion}")
        if season >= 0.7 and intent.season: reasons.append(f"fits {intent.season}")
        if shared_outfit > 0: reasons.append("supported by existing outfit relationships")
        return round(final, 3), {k: round(v, 3) for k, v in breakdown.items()}, total_price, reasons

## Bounded outfit generator

Beam search expands one slot at a time and keeps only the best partial combinations. Optional slots also allow a `None` choice. This prevents uncontrolled Cartesian-product growth.

In [35]:
class CandidateOutfitGenerator:
    def __init__(self, config, scorer):
        self.config = config
        self.scorer = scorer

    def generate(self, candidates, slots, intent):
        beam = [({}, 0.0)]
        evaluated = 0
        pruned_count = 0
        for slot in slots["all"]:
            choices = list(candidates.get(slot, []))
            if slot in slots["optional"]:
                choices = [None] + choices
            if slot in slots["required"] and not choices:
                return [], {"combinations_evaluated": evaluated, "combinations_pruned": pruned_count}
            expanded = []
            limit_reached = False
            for items, _ in beam:
                for choice in choices:
                    if evaluated >= self.config.max_combinations_evaluated:
                        limit_reached = True
                        break
                    evaluated += 1
                    new_items = dict(items)
                    if choice is not None:
                        new_items[slot] = choice
                    if self.config.total_budget_mode and self.config.total_budget is not None:
                        price = sum(x.get("price") or 0.0 for x in new_items.values())
                        if price > self.config.total_budget:
                            pruned_count += 1
                            continue
                    partial_score, _, _, _ = self.scorer.score(new_items, slots["required"], intent,)
                    expanded.append((new_items, partial_score))
                if limit_reached:
                    break
            expanded.sort(key=lambda x: (-x[1], tuple(sorted((k, v["product_id"]) for k, v in x[0].items()))))
            if len(expanded) > self.config.beam_width:
                pruned_count += len(expanded) - self.config.beam_width
            beam = expanded[:self.config.beam_width]

        ranked = []
        for items, _ in beam:
            score, breakdown, total_price, reasons = self.scorer.score(items, slots["required"], intent)
            ranked.append({"items": items, "total_price": total_price, "score": score,
                           "score_breakdown": breakdown, "reasons": reasons})
        ranked.sort(key=lambda x: (-x["score"], x["total_price"], tuple(sorted((k, v["product_id"]) for k, v in x["items"].items()))))
        for rank, outfit in enumerate(ranked[:self.config.top_n], 1):
            outfit["rank"] = rank
        return ranked[:self.config.top_n], {"combinations_evaluated": evaluated, "combinations_pruned": pruned_count}

## Week 6 pipeline

This wrapper reuses Week 5 parsing and category retrieval, plans slots, prunes candidates, builds bounded combinations and returns top-N outfits with diagnostics.

In [36]:
class Week6OutfitPipeline:
    def __init__(self, week5_pipeline, catalog, outfits, config=None):
        self.week5 = week5_pipeline
        self.catalog = catalog
        self.outfits = outfits
        self.config = config or OutfitConstructionConfig()
        self.product_by_id = {str(p["product_id"]): p for p in catalog}
        self.product_outfits = defaultdict(set)
        for outfit in outfits:
            for product_id in outfit.get("product_ids", []):
                self.product_outfits[str(product_id)].add(str(outfit["outfit_id"]))
        self.planner = OutfitSlotPlanner()
        self.pruner = CandidatePruner(self.config)
        self.scorer = OutfitScorer(self.config, self.product_outfits)
        self.generator = CandidateOutfitGenerator(self.config, self.scorer)

    def run(self, query):
        week5_result = self.week5.run(query, candidates_per_category=max(self.config.clothing_limit, 10))
        intent = CustomerIntent.from_dict(week5_result["intent"])
        slots = self.planner.plan(query, intent)

        retrieval_intent = CustomerIntent.from_dict(intent.to_dict())
        retrieval_intent.categories_needed = slots["all"]
        retrieved = self.week5.ranker.retrieve(retrieval_intent, per_category=self.config.retrieval_pool_size, query=query)
        pruned, before, after = self.pruner.prune(retrieved, slots, self.product_by_id)
        outfits_ranked, generation = self.generator.generate(pruned, slots, intent)

        diagnostics = {
            "parser_used": week5_result["parser"],
            "slots_selected": slots,
            "candidates_before_pruning": before,
            "candidates_after_pruning": after,
            **generation,
            "retrieval_pool_size": self.config.retrieval_pool_size,
            "beam_width": self.config.beam_width,
            "maximum_combinations_evaluated": self.config.max_combinations_evaluated,
            "top_n_returned": len(outfits_ranked),
            "budget_mode": "total outfit" if self.config.total_budget_mode else "per product",
            "skipped_filters": ["availability", "gender", "size"],
        }
        return {"query": query, "intent": intent.to_dict(), "slots": slots,
                "candidates": pruned, "top_outfits": outfits_ranked, "diagnostics": diagnostics}


week6_pipeline = Week6OutfitPipeline(pipeline, catalog, outfits, WEEK6_CONFIG)

## Run one complete example

The following cells show the retained candidates per slot and the top-ranked outfit combinations.

In [37]:
QUERY = 'I want something elegant but not too formal for a summer dinner.'
week6_result = week6_pipeline.run(QUERY)

print('STRUCTURED INTENT')
print(json.dumps(week6_result['intent'], indent=2))
print('\nDIAGNOSTICS')
print(json.dumps(week6_result['diagnostics'], indent=2))

STRUCTURED INTENT
{
  "occasion": "summer dinner",
  "style": [
    "elegant",
    "smart casual"
  ],
  "avoid": [
    "too formal"
  ],
  "season": "summer",
  "categories_needed": [
    "tops",
    "bottoms",
    "shoes",
    "bags",
    "accessories"
  ],
  "color_palette": [
    "neutral",
    "light",
    "warm"
  ],
  "price_constraints": null
}

DIAGNOSTICS
{
  "parser_used": "rules_fallback",
  "slots_selected": {
    "required": [
      "tops",
      "bottoms",
      "shoes"
    ],
    "optional": [
      "bags",
      "accessories"
    ],
    "all": [
      "tops",
      "bottoms",
      "shoes",
      "bags",
      "accessories"
    ]
  },
  "candidates_before_pruning": {
    "tops": 20,
    "bottoms": 20,
    "shoes": 20,
    "bags": 20,
    "accessories": 20
  },
  "candidates_after_pruning": {
    "tops": 10,
    "bottoms": 10,
    "shoes": 10,
    "bags": 5,
    "accessories": 5
  },
  "combinations_evaluated": 1430,
  "combinations_pruned": 1180,
  "retrieval_pool_size

In [38]:
candidate_summary = pd.DataFrame([
    {
        'slot': slot,
        'before_pruning': week6_result['diagnostics']['candidates_before_pruning'].get(slot, 0),
        'after_pruning': week6_result['diagnostics']['candidates_after_pruning'].get(slot, 0),
        'retained_item_ids': ', '.join(str(x['product_id']) for x in products),
    }
    for slot, products in week6_result['candidates'].items()
])
display(candidate_summary)

,slot,before_pruning,after_pruning,retained_item_ids
0,tops,20,10,"117635100, 131539979, 146536948, 149722016, 15..."
1,bottoms,20,10,"133066819, 143634795, 144423927, 155010135, 15..."
2,shoes,20,10,"108336922, 115291003, 123260132, 124093373, 12..."
3,bags,20,5,"104834863, 111114607, 115405939, 125227726, 12..."
4,accessories,20,5,"108300637, 119664180, 135755928, 140921645, 14..."


In [39]:
outfit_rows = []
for outfit in week6_result['top_outfits']:
    outfit_rows.append({
        'rank': outfit['rank'],
        'item_ids': ' | '.join(f"{slot}: {p['product_id']}" for slot, p in outfit['items'].items()),
        'titles': ' | '.join(f"{slot}: {p['title']}" for slot, p in outfit['items'].items()),
        'total_price': outfit['total_price'],
        'score': outfit['score'],
        'reasons': '; '.join(outfit['reasons']),
    })
top_outfit_table = pd.DataFrame(outfit_rows)
display(top_outfit_table)

catalog_ids = {str(p['product_id']) for p in catalog}
returned_ids = {str(p['product_id']) for outfit in week6_result['top_outfits'] for p in outfit['items'].values()}
assert returned_ids <= catalog_ids
print(f'Verified {len(returned_ids)} returned item IDs against the real catalog.')

,rank,item_ids,titles,total_price,score,reasons
0,1,tops: 117635100 | bottoms: 143634795 | shoes: ...,tops: rails hunter plaid shirt in pine/white |...,154.49,8.534,complete required slots; compatible color pale...
1,2,tops: 117635100 | bottoms: 143634795 | shoes: ...,tops: rails hunter plaid shirt in pine/white |...,161.47,8.531,complete required slots; compatible color pale...
2,3,tops: 117635100 | bottoms: 192451835 | shoes: ...,tops: rails hunter plaid shirt in pine/white |...,165.94,8.530,complete required slots; compatible color pale...
3,4,tops: 117635100 | bottoms: 143634795 | shoes: ...,tops: rails hunter plaid shirt in pine/white |...,168.92,8.529,complete required slots; compatible color pale...
4,5,tops: 117635100 | bottoms: 192451835 | shoes: ...,tops: rails hunter plaid shirt in pine/white |...,172.92,8.528,complete required slots; compatible color pale...


Verified 7 returned item IDs against the real catalog.


## Automated Week 6 tests

These tests use a small isolated fixture to verify slot planning, duplicate pruning, bounded generation, total-price calculation and real-ID integrity.

In [40]:
def fixture_product(item_id, title, category, color="white", style="elegant", occasion="dinner", season="summer", price=20):
    return normalize_product({
        "item_id": item_id, "title": title, "description": "",
        "refined_category": category, "refined_color": color,
        "material": "cotton", "season": season, "style": style,
        "occasion": occasion, "real_price": price,
    })


fixture_catalog = []
fixture_catalog += [fixture_product(f"T{i}", f"Elegant Top {i}", "tops", color="ivory") for i in range(22)]
fixture_catalog += [fixture_product(f"B{i}", f"Elegant Skirt {i}", "bottoms", color="beige") for i in range(12)]
fixture_catalog += [fixture_product(f"S{i}", f"Elegant Shoe {i}", "shoes") for i in range(12)]
fixture_catalog += [fixture_product(f"G{i}", f"Dinner Bag {i}", "bags", color="black") for i in range(7)]
fixture_catalog += [fixture_product(f"A{i}", f"Gold Necklace {i}", "accessories", color="gold") for i in range(7)]
fixture_catalog += [fixture_product("D1", "Cotton Summer Dress", "dresses", color="beige", price=60)]
fixture_catalog += [fixture_product("O1", "Winter Coat", "outerwear", color="black", season="winter")]

duplicate_a = fixture_product("DU1", "a stylish pair of shoes: White Court Sneakers", "shoes")
duplicate_b = fixture_product("DU2", "White Court Sneaker", "shoes")
assert _duplicate_key(duplicate_a) == _duplicate_key(duplicate_b)

fixture_outfits = [{
    "outfit_id": "SET1", "product_ids": ["T0", "B0", "S0"],
    "style": ["elegant"], "occasion": ["dinner"],
    "season": ["summer"], "colors": ["ivory", "beige", "white"],
}]
fixture_week5 = FashionRAGPipeline(fixture_catalog, fixture_outfits, min_relevance_score=2.0)
fixture_config = OutfitConstructionConfig(
    retrieval_pool_size=20, clothing_limit=10, bag_limit=5, accessory_limit=5,
    beam_width=20, max_combinations_evaluated=500, top_n=3,
)
fixture_pipeline = Week6OutfitPipeline(fixture_week5, fixture_catalog, fixture_outfits, fixture_config)

top_intent = CustomerIntent(style=["elegant"], categories_needed=["tops"])
top_pool = fixture_week5.ranker.retrieve(top_intent, per_category=fixture_config.retrieval_pool_size, query="elegant tops")
assert len(top_pool["tops"]) == 20

separates = fixture_pipeline.run("an elegant summer dinner outfit")
assert separates["slots"]["required"] == ["tops", "bottoms", "shoes"]
assert separates["diagnostics"]["candidates_before_pruning"]["tops"] == 20
assert separates["diagnostics"]["candidates_after_pruning"]["tops"] == 10
assert separates["diagnostics"]["candidates_after_pruning"]["bags"] == 5
assert separates["diagnostics"]["candidates_after_pruning"]["accessories"] == 5
assert separates["top_outfits"]
assert all({"tops", "bottoms", "shoes"} <= set(x["items"]) for x in separates["top_outfits"])

bag_slots = OutfitSlotPlanner().plan("an elegant outfit with a handbag", CustomerIntent(categories_needed=["bags"]))
assert "bags" in bag_slots["required"]
hat_slots = OutfitSlotPlanner().plan("an elegant outfit with a hat", CustomerIntent(categories_needed=["accessories"]))
assert "accessories" in hat_slots["required"]
winter_slots = OutfitSlotPlanner().plan("a winter outfit", CustomerIntent(season="winter"))
assert "outerwear" in winter_slots["optional"]

dress_slots = OutfitSlotPlanner().plan(
    "an elegant cotton dress for a summer dinner",
    CustomerIntent(categories_needed=["dresses"], season="summer", occasion="summer dinner"),
)
assert dress_slots["required"] == ["dresses", "shoes"]
assert "tops" not in dress_slots["all"] and "bottoms" not in dress_slots["all"]

missing_style_a = fixture_product("M1", "Plain Top", "tops", style="")
missing_style_b = fixture_product("M2", "Plain Skirt", "bottoms", style="elegant")
assert _metadata_compatibility(missing_style_a, missing_style_b, "style") == 0.5

known = {p["product_id"] for p in fixture_catalog}
assert {p["product_id"] for x in separates["top_outfits"] for p in x["items"].values()} <= known
assert separates["diagnostics"]["combinations_evaluated"] <= fixture_config.max_combinations_evaluated
assert separates["diagnostics"]["retrieval_pool_size"] == 20
assert separates["diagnostics"]["skipped_filters"] == ["availability", "gender", "size"]
print("All updated Week 6 outfit-construction tests passed.")

All updated Week 6 outfit-construction tests passed.


## Five end-to-end examples

These examples show the complete Week 6 flow from structured intent through slot selection and candidate pruning to top-ranked outfit sets.

In [41]:
EXAMPLE_QUERIES = [
    'I want something elegant but not too formal for a summer dinner.',
    'A casual blue denim outfit for everyday wear under $80.',
    'Minimal white sneakers for summer.',
    'A warm cotton dress for a summer dinner under $100.',
    'Something polished for the office, no leather.',
]

example_results = []
for query in EXAMPLE_QUERIES:
    example = week6_pipeline.run(query)
    example_results.append(example)
    print('\n' + '=' * 100)
    print('CUSTOMER QUERY:', query)
    print('STRUCTURED INTENT:', json.dumps(example['intent'], indent=2))
    print('SLOTS:', json.dumps(example['slots'], indent=2))
    print('DIAGNOSTICS:', json.dumps(example['diagnostics'], indent=2))
    print('TOP OUTFITS:')
    for outfit in example['top_outfits']:
        compact = {
            'rank': outfit['rank'],
            'items': {slot: {'product_id': p['product_id'], 'title': p['title']} for slot, p in outfit['items'].items()},
            'total_price': outfit['total_price'],
            'score': outfit['score'],
            'score_breakdown': outfit['score_breakdown'],
            'reasons': outfit['reasons'],
        }
        print(json.dumps(compact, indent=2))

print(f'\nCompleted {len(example_results)} Week 6 examples.')


CUSTOMER QUERY: I want something elegant but not too formal for a summer dinner.
STRUCTURED INTENT: {
  "occasion": "summer dinner",
  "style": [
    "elegant",
    "smart casual"
  ],
  "avoid": [
    "too formal"
  ],
  "season": "summer",
  "categories_needed": [
    "tops",
    "bottoms",
    "shoes",
    "bags",
    "accessories"
  ],
  "color_palette": [
    "neutral",
    "light",
    "warm"
  ],
  "price_constraints": null
}
SLOTS: {
  "required": [
    "tops",
    "bottoms",
    "shoes"
  ],
  "optional": [
    "bags",
    "accessories"
  ],
  "all": [
    "tops",
    "bottoms",
    "shoes",
    "bags",
    "accessories"
  ]
}
DIAGNOSTICS: {
  "parser_used": "rules_fallback",
  "slots_selected": {
    "required": [
      "tops",
      "bottoms",
      "shoes"
    ],
    "optional": [
      "bags",
      "accessories"
    ],
    "all": [
      "tops",
      "bottoms",
      "shoes",
      "bags",
      "accessories"
    ]
  },
  "candidates_before_pruning": {
    "tops": 20,
 